# 03. Async

**Tier:** Deep. **Why you'd use this:** Many small waits can share one thread if each task yields at `await`. You gather results in order without a thread per wait.

**After this lesson you can:** overlap waiting tasks with `await`; gather their results in order; start async work from a script without nesting a second event loop in a notebook.
About 30 minutes. **Uses:** [Threads](01-threads-lesson.ipynb)


## Quick review

1. When do processes beat threads for speed?
2. Why must a `ProcessPoolExecutor` target be picklable?

<details><summary>Answers</summary>

CPU-bound work that needs more than one core. Workers are separate interpreters that receive the function by pickling.
</details>


## The idea

Async code uses one thread and an event loop. `await` pauses a coroutine until something is ready, letting the loop run other coroutines. `asyncio.gather` starts several awaits together and returns results in input order. Notebooks already have a loop, so you `await` at the top level. Scripts start a loop in a `.py` file (see `run_async_main.py`).

A coroutine function returns a coroutine object when called. The loop drives it until the next `await`. Many coroutines can be waiting on I/O while only one runs Python between awaits. That model fits thousands of open connections better than one thread each. Blocking calls still freeze the loop; wrap them in an executor if you must call them from async code.

<details><summary>Explain more</summary>

Cooperative multitasking: a coroutine keeps the loop busy only until its next `await`. A blocking `time.sleep` inside async code stalls everyone. Use `asyncio.sleep` to wait without blocking the loop.
</details>


## See it work

Fetch labels concurrently, then show how a script starts the loop.


In [1]:
import asyncio


async def fetch(name):
    # 1. Non-blocking wait.
    await asyncio.sleep(0.01)
    return name.upper()


async def fetch_all(names):
    # 2. gather keeps input order.
    return list(await asyncio.gather(*(fetch(name) for name in names)))


print(await fetch_all(["ada", "bea"]))

['ADA', 'BEA']


In [2]:
import subprocess
import sys

print(subprocess.run([sys.executable, "run_async_main.py"], capture_output=True, text=True).stdout)

script loop finished



## Mental model

Async is one clerk with a ticket wheel. Each coroutine takes a number, does a bit of work, and sits down at `await` so the clerk can help the next ticket. Nobody else runs during a blocking call—the clerk is stuck at that window. `asyncio.sleep` is a polite sit-down. `time.sleep` is standing in the doorway.

| API | role |
|---|---|
| `async def` | defines a coroutine function |
| `await` | pause until ready; let others run |
| `gather` | run many; list results in argument order |
| top-level await | notebook drives the current loop |
| `run_async_main.py` | script entry that starts a loop |

Keep CPU bursts short. Offload blocking SDK calls with `asyncio.to_thread` when you must stay in async style. Prefer clear coroutine boundaries over sprinkling bare callbacks. Remember: overlapping waits beat overlapping CPU on a single thread.

When debugging, print timestamps around awaits to see interleaving. If everything suddenly becomes sequential, search for blocking calls. Library docs that say "async native" usually mean you can await their API without a thread.


## Check yourself

Answer these from the example above before you run the cell.


1. Why does the notebook use top-level `await` instead of `asyncio.run`?
2. What stalls the loop: `await asyncio.sleep(1)` or `time.sleep(1)`?
3. Does `gather` preserve argument order or finish order?

<details><summary>Answers</summary>

1. A loop is already running. 2. `time.sleep`. 3. Argument order.
</details>


In [3]:
import asyncio
import time


async def nap(label):
    await asyncio.sleep(0.05)
    return label


start = time.perf_counter()
results = await asyncio.gather(nap("a"), nap("b"), nap("c"))
elapsed = time.perf_counter() - start
print(results)
print(elapsed < 0.12)

['a', 'b', 'c']
True


<details><summary>Check your answers</summary>

`['a', 'b', 'c']` in order. Elapsed under ~0.12s because the sleeps overlap on one thread. CPython detail: timings vary.
</details>


## Why it works this way

The event loop schedules coroutines. `await` yields control when work would wait, so many logical tasks share one thread without creating OS threads per connection. That is cooperative: if you call blocking I/O or `time.sleep`, the loop cannot schedule others until it returns. Deep dive: [The GIL and free-threading](../../internals/12-gil/01-gil-lesson.ipynb) for how threads differ; async avoids thread switching costs when waits dominate.

`asyncio.gather` registers the awaitables and resolves to a list in the same order as the arguments, even if completions differ. Notebooks embed a running loop, so `asyncio.run` raises if you call it here; scripts use `run_async_main.py` instead. AGENTS rule: never put `asyncio.run` in a notebook cell. Top-level `await` is the notebook form of "drive this coroutine on the current loop."

Cost per task is a small coroutine object plus whatever buffers the I/O stack needs. Thousands of waiting sockets can fit where threads would exhaust memory. CPU-heavy work still belongs in processes; running a tight loop in a coroutine starves peers the same way `time.sleep` does. Cancellation and timeouts need await points to take effect. Design reason for async/await syntax: make yield points visible so the loop can interleave.

Evidence: overlapping `asyncio.sleep` via top-level await finishes near the max delay, not the sum; ordered `gather` results; a subprocess script that prints `shorter.loop finished` style output from `run_async_main.py`. Trace: start three naps, each awaits sleep, loop runs them concurrently, gather rebuilds the list as `['a','b','c']`.


In [4]:
import asyncio
import time


async def nap(label):
    await asyncio.sleep(0.05)
    return label


start = time.perf_counter()
print(await asyncio.gather(nap("a"), nap("b")))
print(time.perf_counter() - start < 0.1)

['a', 'b']
True


## When to use it

Use async for many concurrent waits (sockets, timers) in one thread. Use threads when libraries are blocking-only. Use processes for CPU. Don't call blocking functions inside coroutines. Don't call `asyncio.run` inside a notebook.

Reach for async when you own the I/O and need many concurrent waits. Prefer threads when a mature blocking SDK is the whole story. Prefer processes for CPU. Avoid mixing heavy CPU into the loop. Avoid nesting `asyncio.run` calls. Prefer `TaskGroup` when child lifetimes should nest with a block.


## Common mistake

Using `time.sleep` inside a coroutine and wondering why other tasks freeze.


In [5]:
import asyncio
import time


async def bad():
    time.sleep(0.05)
    return "done"


async def other():
    return "other"


start = asyncio.get_running_loop().time()
results = await asyncio.gather(bad(), other())
# other cannot run during time.sleep
print(results)

['done', 'other']


Replace `time.sleep` with `await asyncio.sleep` so the loop can interleave.


In [6]:
import asyncio


async def good():
    await asyncio.sleep(0.05)
    return "done"


print(await good())

done


## Try it

Change `fetch_all` to uppercase only names that start with `"a"`. Where does the filter belong if you still want concurrent fetches for the rest?


## Recap

- Async overlaps waits on one thread with `await`.
- `gather` returns results in input order.
- Scripts start the loop in a `.py` file; notebooks use top-level await.

**Next:** [practice](03-async-practice.ipynb)
